<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_6_8_%D0%90%D0%B2%D1%82%D0%BE%D0%BD%D0%BE%D0%BC%D0%BD%D1%8B%D0%B5_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D1%8B_%D0%BF%D0%BB%D0%B0%D0%BD%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B8_%D1%81%D0%B0%D0%BC%D0%BE%D0%BE%D1%86%D0%B5%D0%BD%D0%BA%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.8. Автономные агенты: планирование и самооценка

## Введение: от реактивных помощников к проактивным агентам

Поздравляю! Мы прошли невероятный путь. В Лекции 6.1 мы создавали простого RAG-агента, который отвечал на вопросы по документам. В Лекции 6.4 мы научили агента вызывать инструменты и принимать решения. В Лекции 6.6 мы построили команду специализированных агентов, а в Лекции 6.7 добавили супервайзера, который управляет этой командой. Но все эти агенты были **реактивными** – они получали один вопрос, выполняли несколько шагов и останавливались. Они не планировали долгосрочные действия, не оценивали свой прогресс и не корректировали план на ходу.

Теперь мы подходим к **вершине** – автономным агентам. Представьте, что вы даёте агенту не вопрос, а **цель**: «Напиши аналитический отчёт о состоянии рынка ИИ в 2026 году». Агент сам:
1. Разбивает задачу на подзадачи (собрать данные, проанализировать, написать текст, отредактировать).
2. Выполняет их последовательно, используя доступные инструменты.
3. Проверяет качество своей работы.
4. Если результат неудовлетворительный – возвращается и исправляет ошибки.

Это и есть **автономный агент** – система, которая самостоятельно планирует, действует, оценивает и адаптируется до тех пор, пока цель не будет достигнута. Это уже не просто «ответ на вопрос», а настоящий цифровой помощник, способный решать сложные многошаговые задачи.

В этой лекции мы реализуем такого агента. Мы объединим три ключевых паттерна:
- **ReAct (Reasoning + Acting)** – агент чередует размышления и действия.
- **Plan‑and‑Execute** – агент сначала составляет план, потом выполняет.
- **Reflexion** – агент оценивает свои результаты и учится на ошибках.

К концу лекции вы получите полностью автономного агента, который сможет самостоятельно выполнять сложные задачи – от написания отчётов до проведения исследований. Поехали!

---

## Тема 1. Что такое автономный агент и зачем он нужен

### 1.1. Отличие от реактивного агента

Все агенты, которые мы строили ранее, были **реактивными**. Они получали запрос и выполняли заранее определённую последовательность действий. Даже супервайзер из Лекции 6.7, хотя и принимал решения на каждом шаге, делал это в рамках одной задачи. Как только ответ был сгенерирован – работа завершалась.

**Автономный агент** работает иначе:

| Аспект | Реактивный агент | Автономный агент |
|--------|------------------|------------------|
| **Инициатива** | Отвечает на запрос | Самостоятельно ставит подцели |
| **Планирование** | Фиксированный порядок шагов | Динамический план, который может меняться |
| **Оценка** | Не проверяет качество | Проверяет результат и исправляет ошибки |
| **Остановка** | Останавливается после ответа | Работает до достижения цели |
| **Адаптивность** | Не меняет стратегию | Меняет план при неудачах |

**Пример:** если реактивному агенту сказать «напиши отчёт», он может просто сгенерировать текст на основе имеющихся данных. Автономный агент сначала подумает: «Что нужно для отчёта? Какие данные собрать? Где их взять? Проверить ли факты?» – и только потом начнёт действовать, постоянно оценивая прогресс.

### 1.2. Примеры задач для автономных агентов

Автономные агенты особенно полезны там, где задача не может быть решена за один шаг:

- **Написание аналитического отчёта** – сбор данных, анализ, структурирование, написание, редактура.
- **Исследование темы** – поиск информации, проверка источников, синтез знаний, формулировка выводов.
- **Автоматизация рутины** – обработка писем, планирование встреч, управление проектами.
- **Обучение и саморазвитие** – агент изучает новую тему и проверяет свои знания.
- **Программирование** – написание кода, тестирование, отладка, рефакторинг.

В каждом из этих сценариев агенту нужно не просто ответить, а **достичь цели** – и для этого требуется планирование, оценка и адаптация.

### 1.3. Основные компоненты автономного агента

Автономный агент состоит из трёх ключевых компонентов, работающих в цикле:

1. **Планировщик (Planner)** – разбивает цель на подзадачи и определяет порядок их выполнения.
2. **Исполнитель (Executor)** – выполняет подзадачи, используя доступные инструменты.
3. **Оценщик (Evaluator / Self‑Critic)** – анализирует результат, проверяет, достигнута ли цель, и решает, нужно ли корректировать план.

К этим трём добавляется **память** – не только краткосрочная (история диалога), но и долгосрочная (сохранение результатов предыдущих шагов, чтобы не повторять их).

**Схема работы:**

```
Пользователь задаёт цель
        ↓
┌───────────────────────────────────────┐
│  Планировщик (LLM)                    │
│  "Что нужно сделать для достижения?   │
│   Какой следующий шаг?"               │
└──────────────────┬────────────────────┘
                   ↓
┌───────────────────────────────────────┐
│  Исполнитель (Executor)               │
│  Выполняет шаг (вызов инструмента,    │
│  поиск, вычисление)                   │
└──────────────────┬────────────────────┘
                   ↓
┌───────────────────────────────────────┐
│  Оценщик (Self‑Critic)                │
│  "Достигнута ли цель? Нужно ли        │
│   изменить план?"                     │
└──────────────────┬────────────────────┘
                   ↓
        ┌──────────┴──────────┐
        │  Цель достигнута?    │
        │  Да → Ответ          │
        │  Нет → Вернуться к   │
        │        планировщику  │
        └─────────────────────┘
```

Каждый из этих компонентов может быть реализован как отдельный LLM-агент или как один агент, который выполняет все три роли в цикле.

### 1.4. Обзор подходов к автономности

Существует несколько популярных паттернов для построения автономных агентов:

#### ReAct (Reasoning + Acting)

Агент чередует «размышление» и «действие». На каждом шаге он пишет, что собирается сделать, выполняет действие, анализирует результат и решает, что делать дальше.

```
Шаг 1: "Мне нужно найти информацию о компании X" → поиск
Шаг 2: "Теперь нужно сравнить с компанией Y" → поиск
Шаг 3: "Данные собраны, можно писать ответ" → генерация
```

Этот паттерн мы уже использовали в Лекции 6.4 – агент с инструментами по сути работал по схеме ReAct.

#### Plan‑and‑Execute

Агент сначала составляет полный план действий, а затем последовательно его выполняет. Это делает поведение более предсказуемым и позволяет видеть весь маршрут заранее.

```
План:
1. Найти информацию о компании X
2. Найти информацию о компании Y
3. Сравнить показатели
4. Написать отчёт
5. Проверить факты
→ Выполнение по шагам
```

#### Reflexion (Самооценка)

Агент генерирует ответ, затем критикует его, указывает на ошибки и генерирует исправленный вариант. Этот цикл повторяется, пока качество не станет удовлетворительным.

```
Шаг 1: "Вот мой ответ..." (генерация)
Шаг 2: "Я ошибся в датах, нужно исправить" (рефлексия)
Шаг 3: "Вот исправленный ответ" (новая генерация)
```

#### Tree of Thoughts (Дерево мыслей)

Агент генерирует несколько вариантов решения, оценивает каждый и выбирает лучший. Это похоже на то, как человек рассматривает разные варианты перед принятием решения.

В этой лекции мы объединим **Plan‑and‑Execute** и **Reflexion**, чтобы получить максимально автономного агента.

### 1.5. Какие пакеты нужны

Для реализации автономного агента нам не понадобятся новые библиотеки. Всё, что мы использовали раньше, остаётся:

```bash
pip install langchain langchain-ollama langgraph chromadb sentence-transformers
```

Мы будем использовать:
- **LangGraph** – для построения графа с циклом (планировщик → исполнитель → оценщик → планировщик).
- **LangChain** – для работы с LLM и инструментами.
- **Ollama** – как локальный сервер для LLM.
- **Chroma** – для долгосрочной памяти (сохранение результатов шагов).

Никаких дополнительных установок не требуется – всё уже есть в вашем проекте.

---

## Краткий итог Тема 1

- **Автономный агент** – это система, которая самостоятельно планирует, действует, оценивает и адаптируется до достижения цели.
- Он отличается от реактивного агента **проактивностью** – он не просто отвечает, а сам ставит подцели и корректирует стратегию.
- **Три ключевых компонента**: планировщик, исполнитель, оценщик.
- **Основные паттерны**: ReAct, Plan‑and‑Execute, Reflexion, Tree of Thoughts.
- Мы будем использовать **LangGraph** для реализации цикла и **Ollama** для LLM.

---

**В следующей теме мы перейдём к реализации планировщика и настроим цикл «план → действие → оценка».**